# Here, we will fine-tune a LLAMA model to prevent overfiltering

Here, we import all the libraries that we need to run the code

In [ ]:
import json
import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

## Now, we load the dataset to fine-tune model on it

In [ ]:
with open("train.json") as f:
    raw_data = json.load(f)

records = [{"text": d["prompt"], "label": int(d["label"])} for d in raw_data]

dataset = Dataset.from_list(records)
dataset = dataset.class_encode_column("label")
split = dataset.train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
train_ds, eval_ds = split["train"], split["test"]

print(f"Train size: {len(train_ds)}  |  Eval size: {len(eval_ds)}")

## Use this bloct to load the tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)


# keep the raw text length around so group_by_length can bucket similar-length
# examples together (your data ranges from a few words to 8000+ words, so this
# avoids wasting huge amounts of compute padding short sequences up to long ones)
train_ds = train_ds.map(tokenize_fn, batched=True)
eval_ds = eval_ds.map(tokenize_fn, batched=True)
train_ds = train_ds.map(lambda x: {"length": len(x["input_ids"])})
eval_ds = eval_ds.map(lambda x: {"length": len(x["input_ids"])})
train_ds = train_ds.remove_columns(["text"])
eval_ds = eval_ds.remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Load the model

In [ ]:
quant_config = None
USE_4BIT = True
if USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForSequenceClassification.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    num_labels=2,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.pad_token_id = tokenizer.pad_token_id

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Here, you will show the metrics that to need for prediction and eval

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

## Use code below for finr-tune

In [ ]:
from transformers import TrainerCallback

class LogHistoryCallback(TrainerCallback):
    def on_log(self, args, state, control, **kwargs):
        with open(f"{OUTPUT_DIR}/log_history_live.json", "w") as f:
            json.dump(state.log_history, f, indent=2)

OUTPUT_DIR = "/content/drive/MyDrive/NLPsave/weightsSave"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    optim="paged_adamw_32bit",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.06,
    max_grad_norm=0.3,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=4,  
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    length_column_name="length",
    dataloader_pin_memory=False,
    report_to="none",
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[LogHistoryCallback()],
)

import os

last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        last_checkpoint = os.path.join(
            OUTPUT_DIR, sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        )
        print(f"Resuming from {last_checkpoint}")
    else:
        print("No checkpoint found, starting fresh")

trainer.train(resume_from_checkpoint=last_checkpoint)

## Save Lora Adapter

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")

print("Done. Adapter saved to", f"{OUTPUT_DIR}/final_adapter")